# CEHARPS 02 — เตรียมข้อมูลสุขภาพและแบ่งเชิงพื้นที่ / Prepare and spatially split health data

ส่วนนี้ไม่ใช้ GMW หรือหน้ากากป่าชายเลนเป็นตัวแปรสุขภาพ และแบ่ง train/validation/test ด้วย `spatial_block` เพื่อป้องกันพื้นที่ใกล้กันรั่วข้ามชุดข้อมูล


In [ ]:
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import subprocess  # TH: นำเข้าเครื่องมือเรียกคำสั่งระบบ | EN: Import subprocess utilities.
import sys  # TH: นำเข้าข้อมูลตัวแปลภาษา Python | EN: Import Python runtime information.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn>=1.5,<2"])  # TH: ติดตั้ง scikit-learn รุ่นที่รองรับ | EN: Install a supported scikit-learn version.
import numpy as np  # TH: นำเข้า NumPy | EN: Import NumPy.
import pandas as pd  # TH: นำเข้า pandas | EN: Import pandas.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
from sklearn.model_selection import GroupShuffleSplit  # TH: นำเข้าเครื่องมือแบ่งข้อมูลตามกลุ่ม | EN: Import group-aware splitting.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลาง | EN: Load shared settings.
SEED = int(CONFIG["seed"])  # TH: อ่านค่าเมล็ดสุ่ม | EN: Read the random seed.
TARGET = str(CONFIG["health_target"])  # TH: อ่านชื่อเป้าหมายสุขภาพ | EN: Read the health target name.


In [ ]:
mode = str(CONFIG["health_mode"]).lower()  # TH: อ่านโหมดข้อมูลสุขภาพ | EN: Read the health-data mode.
real_path = Path(CONFIG["health_real_csv"])  # TH: อ่านพาธข้อมูลสุขภาพจริง | EN: Read the real health-data path.
demo_path = PROJECT_ROOT / "data/health/health_features_demo_SYNTHETIC.csv"  # TH: กำหนดพาธข้อมูลจำลอง | EN: Define the synthetic-data path.
csv_path = real_path if mode == "real" else demo_path  # TH: เลือกไฟล์ตามโหมด | EN: Select the file based on mode.
if not csv_path.exists():  # TH: ตรวจว่าไฟล์มีอยู่หรือไม่ | EN: Check whether the selected file exists.
    raise FileNotFoundError(f"Missing health dataset: {csv_path}")  # TH: หยุดเมื่อไม่พบข้อมูล | EN: Stop when the dataset is missing.
frame = pd.read_csv(csv_path)  # TH: อ่านข้อมูลสุขภาพ | EN: Load the health table.
required = {"sample_id", "site_id", "spatial_block", TARGET}  # TH: กำหนดคอลัมน์ขั้นต่ำ | EN: Define minimum required columns.
missing = required.difference(frame.columns)  # TH: หาคอลัมน์ที่ขาด | EN: Find missing columns.
if missing:  # TH: ตรวจว่ามีคอลัมน์ขาดหรือไม่ | EN: Check for missing columns.
    raise ValueError(f"Missing required columns: {sorted(missing)}")  # TH: แจ้งคอลัมน์ที่ขาด | EN: Report missing columns.
if not pd.api.types.is_numeric_dtype(frame[TARGET]):  # TH: ตรวจว่าเป้าหมายเป็นตัวเลข | EN: Check that the target is numeric.
    raise TypeError(f"{TARGET} must be numeric")  # TH: หยุดเมื่อชนิดข้อมูลผิด | EN: Stop on an invalid target type.
if not frame[TARGET].dropna().between(0, 100).all():  # TH: ตรวจช่วงคะแนน 0–100 | EN: Validate the 0–100 target range.
    raise ValueError(f"{TARGET} must be between 0 and 100")  # TH: หยุดเมื่อคะแนนอยู่นอกช่วง | EN: Stop when target values are outside range.
frame = frame.dropna(subset=[TARGET, "spatial_block"]).reset_index(drop=True)  # TH: ตัดแถวที่ไม่มีเป้าหมายหรือกลุ่มพื้นที่ | EN: Drop rows without targets or spatial groups.
excluded = {"sample_id", "site_id", "spatial_block", "split", "data_status", "MHI", "SHI", "CHI"}  # TH: กำหนดคอลัมน์ที่ห้ามเป็นตัวแปรอิสระ | EN: Define columns excluded from features.
feature_columns = [column for column in frame.select_dtypes(include=[np.number]).columns if column not in excluded]  # TH: เลือกตัวแปรอิสระเชิงตัวเลข | EN: Select numeric predictor columns.
forbidden = {column for column in feature_columns if "gmw" in column.lower() or "habitat_label" in column.lower() or "mangrove_mask" in column.lower()}  # TH: ตรวจป้ายถิ่นที่อยู่ที่อาจทำให้ข้อมูลรั่ว | EN: Detect habitat labels that could leak information.
if forbidden:  # TH: ตรวจว่าพบตัวแปรต้องห้ามหรือไม่ | EN: Check for forbidden predictors.
    raise ValueError(f"Habitat labels cannot be health features: {sorted(forbidden)}")  # TH: หยุดเพื่อป้องกัน label leakage | EN: Stop to prevent label leakage.
if not feature_columns:  # TH: ตรวจว่ามีตัวแปรอิสระหรือไม่ | EN: Check that predictors exist.
    raise ValueError("No numeric feature columns found")  # TH: หยุดเมื่อไม่มีตัวแปรฝึก | EN: Stop when no training features are available.


In [ ]:
groups = frame["spatial_block"].astype(str)  # TH: แปลงรหัสบล็อกเป็นข้อความ | EN: Convert spatial-block IDs to text.
if groups.nunique() < 6:  # TH: ตรวจจำนวนบล็อกขั้นต่ำ | EN: Validate the minimum block count.
    raise ValueError("At least 6 spatial blocks are required")  # TH: หยุดเมื่อกลุ่มพื้นที่น้อยเกินไป | EN: Stop when too few spatial groups exist.
first = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)  # TH: เตรียมแยก train 70% กับชุดชั่วคราว 30% | EN: Prepare a 70/30 group split.
train_index, temporary_index = next(first.split(frame, groups=groups))  # TH: แบ่งตามบล็อกครั้งแรก | EN: Perform the first group-aware split.
temporary = frame.iloc[temporary_index].copy()  # TH: สร้างชุดชั่วคราว | EN: Build the temporary subset.
temporary_groups = temporary["spatial_block"].astype(str)  # TH: อ่านกลุ่มของชุดชั่วคราว | EN: Read temporary-set groups.
second = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED + 1)  # TH: เตรียมแบ่ง val กับ test เท่า ๆ กัน | EN: Prepare an equal validation/test split.
val_local, test_local = next(second.split(temporary, groups=temporary_groups))  # TH: แบ่งชุดชั่วคราวตามบล็อก | EN: Split the temporary set by group.
frame["split"] = ""  # TH: สร้างคอลัมน์ผลการแบ่ง | EN: Initialize the split column.
frame.loc[frame.index[train_index], "split"] = "train"  # TH: ติดป้ายชุดฝึก | EN: Label training rows.
frame.loc[temporary.index[val_local], "split"] = "val"  # TH: ติดป้ายชุดตรวจสอบ | EN: Label validation rows.
frame.loc[temporary.index[test_local], "split"] = "test"  # TH: ติดป้ายชุดทดสอบ | EN: Label test rows.
block_sets = {name: set(frame.loc[frame["split"] == name, "spatial_block"].astype(str)) for name in ["train", "val", "test"]}  # TH: สร้างชุดบล็อกของแต่ละส่วน | EN: Build spatial-block sets for each split.
assert block_sets["train"].isdisjoint(block_sets["val"])  # TH: ยืนยัน train ไม่ทับ val | EN: Confirm train and validation blocks are disjoint.
assert block_sets["train"].isdisjoint(block_sets["test"])  # TH: ยืนยัน train ไม่ทับ test | EN: Confirm train and test blocks are disjoint.
assert block_sets["val"].isdisjoint(block_sets["test"])  # TH: ยืนยัน val ไม่ทับ test | EN: Confirm validation and test blocks are disjoint.
processed = PROJECT_ROOT / "data/processed"  # TH: กำหนดโฟลเดอร์ข้อมูลพร้อมฝึก | EN: Define the processed-data folder.
processed.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์ข้อมูลพร้อมฝึก | EN: Create the processed-data folder.
for split in ["train", "val", "test"]:  # TH: วนบันทึกข้อมูลแต่ละชุด | EN: Iterate through each split for saving.
    frame.loc[frame["split"] == split].to_csv(processed / f"health_{split}.csv", index=False)  # TH: บันทึกชุดข้อมูลปัจจุบัน | EN: Save the current split.
metadata = {"source": str(csv_path), "health_mode": mode, "target": TARGET, "features": feature_columns, "split_method": "spatial_group", "seed": SEED, "warning": "demo data is synthetic" if mode == "demo" else "real data requires provenance review"}  # TH: สร้างเมทาดาทาสำหรับตรวจสอบย้อนกลับ | EN: Build traceability metadata.
(processed / "health_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")  # TH: บันทึกเมทาดาทา | EN: Save the metadata.
print(frame.groupby("split").size())  # TH: แสดงจำนวนแถวแต่ละชุด | EN: Display split row counts.
print("Features:", feature_columns)  # TH: แสดงตัวแปรที่ใช้ฝึก | EN: Display selected features.
